In [ ]:
import os
from collections import defaultdict

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from tqdm.auto import tqdm
from itertools import product

from utils.helpers import list_year_rasters
from utils.helpers import window_grid
from utils.helpers import total_windows
from utils.helpers import pixel_area_hectares

## Define the paths necessary for the logic

In [ ]:
# Define a path to the project folder
project_folder = "<PATH/TO/PROJECT/FOLDER>"

# Define the path to folder containing severity rasters
ras_dir = f"{project_folder}/data/rasters/mtbs_severity_annual_masked"

# Define a path to the raster containing the NFS access class proportions
access_rast_path = f"{project_folder}/data/rasters/nfs_access_proportions.tif"

# Define a path to the raster containing the state integer IDs
state_rast_path  = f"{project_folder}/data/rasters/western_states.tif"

# Define a path to the 
state_shp_path =  f"{project_folder}/data/shapefiles/westernstates.shp"

# Define the output directory
out_dir = f"{project_folder}/data/tables"

# Define the output file path
out_csv = os.path.join(out_dir, "mtbs_nfs_access_class_summary.csv")


## Get the state name lookup table

In [ ]:
# Read in the States shapefile and get the values needed to remap the
# integer values to strings
states = gpd.read_file(state_shp_path)[["STATE_INT", "NAME"]]

# Get the state name lookup table
state_lookup_table = dict(zip(states["STATE_INT"].astype(int), states["NAME"]))


## Extract the raster statistics

In [ ]:
# List rasters
years = list_year_rasters(ras_dir)

# Open access+state rasters once
with rasterio.open(access_rast_path) as dz, rasterio.open(state_rast_path) as ds:

    # Sanity: access proportions must be 3 bands
    if dz.count != 3:
        raise ValueError(f"{os.path.basename(access_rast_path)} must have 3 bands (roadless, wilderness, roaded). Found {dz.count}.")

    # Derive access band names from descriptions (fallback if missing)
    band_names = list(dz.descriptions) if dz.descriptions and any(dz.descriptions) else ["roadless", "wilderness", "roaded"]
    if len(band_names) != 3:
        band_names = ["roadless", "wilderness", "roaded"]

    blockx = 4096
    blocky = 4096

    pix_ha = pixel_area_hectares(dz)
    grid_transform = dz.transform
    grid_shape = (dz.height, dz.width)
    grid_crs = dz.crs

    # Enforce alignment across access/state rasters
    if (ds.crs != grid_crs or ds.transform != grid_transform or ds.width != grid_shape[1] or ds.height != grid_shape[0]):
        raise ValueError("Grid mismatch between access and state rasters. Ensure both were created from the same template.")

    rows = []

    # Outer progress over years
    for year, rpath in tqdm(years, total=len(years), desc="Years", unit="year"):
        
        # Open severity
        with rasterio.open(rpath) as dr:
            
            # Alignment checks vs template grid
            if (dr.crs != grid_crs or dr.transform != grid_transform
                    or dr.width != grid_shape[1] or dr.height != grid_shape[0]):
                raise ValueError(
                    f"Grid mismatch for {os.path.basename(rpath)}. "
                    "Ensure severity rasters share the same template."
                )

            # Count the total number of windows that will need to be iterated over
            w_total = total_windows(dr.width, dr.height, blockx, blocky)

            # Accumulator: (state_int, sev_class, band_idx) -> sum of proportions (0..100 units)
            acc = defaultdict(float)

            # Blockwise progress
            for win in tqdm(
                window_grid(dr.width, dr.height, blockx, blocky),
                total=w_total, leave=False, desc=f"Blocks {year}", mininterval=2
                ):

                # Read in the severity data -- skip tile if no MTBS data is present
                sev = dr.read(1, window=win, masked=True)
                if np.all((sev.data == -1) | (sev.data == 0)):
                    continue
                    
                # Read 3-band access proportions (0-100)
                # Skip tile if all NFS access catagories are equal to 0
                access = dz.read(indexes=[1, 2, 3], window=win, masked=False)
                if not access.any():  
                    continue
                    
                # Read in the state data
                state = ds.read(1, window=win, masked=False)

                # Balid where severity is real, not -1, and state > 0
                valid = (~sev.mask) & (sev.data != -1) & (state != 0)
                if not np.any(valid):
                    continue

                # Flatten valid indices
                sev_v = sev.data[valid].astype(np.int32)
                st_v = state[valid].astype(np.int32)

                # Pack (state, sev) to group efficiently within the block
                comp = st_v * 1000 + sev_v  # assumes sev_class < 1000
                comp_u, inv = np.unique(comp, return_inverse=True)

                # For each access band, sum proportional weights per (state, sev) group
                for b in range(3):
                    z = access[b][valid].astype(np.float64)  # weights: 0..100 per pixel
                    if np.all(z == 0):
                        continue
                    sums = np.bincount(inv, weights=z, minlength=len(comp_u))
                    if sums.size == 0:
                        continue

                    # Update global accumulator
                    for i, c in enumerate(comp_u):
                        wsum = sums[i]
                        if wsum == 0.0:
                            continue
                        st = int(c // 1000)
                        sc = int(c % 1000)
                        acc[(st, sc, b)] += wsum

        # Flush accumulations for this year into rows
        for (st, sc, b), sum_prop in acc.items():
            
            # Sum_prop is sum of 0..100 percentages; convert to area via (sum/100) * pix_area
            area_ha = (sum_prop * pix_ha) / 100.0
            rows.append({
                "year": year,
                "mtbs_class": int(sc),
                "access": band_names[b],
                "state_int": int(st),
                "area_ha": area_ha
            })
            

## Save the raster statistics

In [ ]:
# Convert accumulated results to DataFrame
rows_df = pd.DataFrame(rows)

# Add computed fields and lookup state names
rows_df["area_km2"] = rows_df["area_ha"] / 100.0
rows_df["state_name"] = rows_df["state_int"].astype(int).map(state_lookup_table)
rows_df = rows_df[["state_name", "year", "access", "mtbs_class", "area_km2"]]

# Define the complete domain for all categorical dimensions
all_years   = [int(y) for y, _ in years]  # years processed
state_names = states["NAME"].tolist()     # all states from shapefile
access_cats = band_names                  # access categories from raster

# Use observed severity classes if available, otherwise default range
sev_classes = (sorted(rows_df["mtbs_class"].unique().tolist())
               if not rows_df.empty else [0, 1, 2, 3, 4])

# Create scaffold: all combinations of state × year × access × severity
# This ensures zero-area entries exist for combinations where no fire occurred
scaffold = pd.DataFrame(
    product(state_names, all_years, access_cats, sev_classes),
    columns = ["state_name", "year", "access", "mtbs_class"]
)

# Merge actual fire data onto scaffold, filling gaps with zero area
df = scaffold.merge(rows_df, how = "left", on = ["state_name", "year", "access", "mtbs_class"])
df["area_km2"] = df["area_km2"].fillna(0.0)

# Sort for readability and write to CSV
df = df.sort_values(["state_name", "year", "access", "mtbs_class"]).reset_index(drop=True)
df.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv}")